# 数据结构教程：图（Graph）——从存储到遍历

欢迎来到数据结构中最激动人心的章节——**图（Graph）**。在此之前，你学过的线性表、栈、队列是"一对一"的关系，树是"一对多"的关系，而图则是最一般的"**多对多**"关系。现实世界中的社交网络、地图导航、网页链接、课程依赖……几乎都可以用图来建模。

本教程将带你从零开始，由浅入深地掌握图的核心知识。

---

## 第一部分：建立直觉——什么是图？

---

### 1. 图的基本概念（6.1.1）

#### 1.1 图的定义

**图 (Graph)** 由两个集合组成：

- **顶点集 V (Vertex)**：一组对象，称为顶点或节点。
- **边集 E (Edge)**：顶点之间的连接关系。

用数学语言表示：**G = (V, E)**

```
举例：一个社交网络
    V = {Alice, Bob, Charlie, David}
    E = {(Alice,Bob), (Bob,Charlie), (Alice,David)}

图示：
    Alice ---- Bob
      |          |
      |          |
    David    Charlie
```

> **与树的关系**：树是图的一种特例——树是**连通的、无环的无向图**。所以你之前学的树的知识，全部可以视为图的子集。

#### 1.2 图的基本术语

| 术语 | 含义 | 举例 |
|------|------|------|
| **顶点 (Vertex/Node)** | 图中的数据元素 | 城市、人 |
| **边 (Edge)** | 两个顶点之间的连接 | 公路、好友关系 |
| **有向图 (Directed Graph / Digraph)** | 边有方向，用 `<u, v>` 表示从 u 到 v | 微博关注（我关注你，你不一定关注我） |
| **无向图 (Undirected Graph)** | 边没有方向，用 `(u, v)` 表示 | 微信好友（互相的） |
| **权 (Weight)** | 边上附带的数值 | 两城市间的距离 |
| **带权图/网 (Weighted Graph/Network)** | 边带权值的图 | 地图上的道路网 |

```
无向图示例：            有向图示例：
  A --- B               A --→ B
  |     |               ↑     |
  |     |               |     ↓
  D --- C               D ←-- C
```

#### 1.3 图的更多术语

**（1）邻接 (Adjacent)**

- **无向图**：若存在边 `(u, v)`，则称 u 和 v **互为邻接点**。
- **有向图**：若存在弧 `<u, v>`，则称 u **邻接到** v，v **邻接自** u。

**（2）度 (Degree)**

- **无向图**：顶点 v 的**度** `TD(v)` = 与 v 相连的边数。
- **有向图**：
  - **入度** `ID(v)` = 以 v 为终点的弧的数目
  - **出度** `OD(v)` = 以 v 为起点的弧的数目
  - **度** `TD(v) = ID(v) + OD(v)`

```
有向图示例：
    A --→ B --→ C
    ↑           |
    └-----------┘

顶点A: 入度=1(C→A), 出度=1(A→B), 度=2
顶点B: 入度=1(A→B), 出度=1(B→C), 度=2
顶点C: 入度=1(B→C), 出度=1(C→A), 度=2
```

> **核心公式**：对于任意图，**所有顶点的度之和 = 2 × 边数**。
> 
> 直觉：每条边贡献两个"端点"，所以度总和一定是边数的两倍。

**（3）路径 (Path) 与路径长度**

- **路径**：从顶点 u 到顶点 v 经过的顶点序列。
- **路径长度**：路径上边的数目（带权图则为权值之和）。
- **简单路径**：路径中顶点不重复出现。
- **回路 (Cycle)**：起点和终点相同的路径。
- **简单回路**：除首尾外，顶点不重复的回路。

**（4）连通性**

| 概念 | 适用 | 含义 |
|------|------|------|
| **连通 (Connected)** | 无向图 | 顶点 u 到 v 有路径 |
| **连通图** | 无向图 | 任意两个顶点都连通 |
| **连通分量** | 无向图 | 极大连通子图 |
| **强连通** | 有向图 | u 到 v **和** v 到 u 都有路径 |
| **强连通图** | 有向图 | 任意两顶点都强连通 |
| **强连通分量** | 有向图 | 极大强连通子图 |

```
连通图：              非连通图（3个连通分量）：
  A---B---C             A---B     D---E
  |       |                       |
  D-------E             C         F
```

**（5）子图 & 生成子图**

- **子图**：V' ⊆ V，E' ⊆ E 构成的图。
- **生成子图 (Spanning Subgraph)**：包含原图所有顶点的子图。
- **生成树**：连通图的一个生成子图，且是树（n个顶点，n-1条边，无环，连通）。

**（6）完全图**

- **无向完全图**：任意两顶点之间都有边。n 个顶点有 **n(n-1)/2** 条边。
- **有向完全图**：任意两顶点之间都有方向相反的两条弧。n 个顶点有 **n(n-1)** 条弧。

```
无向完全图 K₄：         有向完全图 K₃：
    A---B                 A ⇄ B
    |\ /|                 ↕ ╲╱ ↕
    | X |                   ╳
    |/ \|                 ↕ ╱╲ ↕
    D---C                 C ⇄ (通过A,B)
```

**（7）稠密图 vs 稀疏图**

- **稠密图**：边数接近完全图，E 接近 V²。
- **稀疏图**：边数远少于完全图，E 远小于 V²。
- 经验法则：若 |E| < |V| × log|V|，通常认为是稀疏图。

> 这个区分非常重要——**稠密图用邻接矩阵存，稀疏图用邻接表存**，这是后面学习的核心判断依据。

---

## 第二部分：图的存储表示

图的"存储"是核心中的核心。不同的存储方式，决定了不同操作的效率。我们将依次讲解五种表示方法。

---

### 2. 边集数组（Edge List）

#### 2.1 思想

最直觉的方法：直接用一个数组/列表，把所有的边存下来。

```
每条边存为一个结构体：{起点, 终点, 权重(可选)}
```

#### 2.2 C++ 实现

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

// 边的定义

In [ ]:
struct Edge {
    int src;    // 起点
    int dest;   // 终点
    int weight; // 权重（无权图可省略）
};

In [ ]:
// 边集数组表示的图
struct EdgeListGraph {
    int V;              // 顶点数
    vector<Edge> edges; // 所有边
};

In [ ]:
int main() {
    // 创建一个 4 个顶点的图
    // 0---1
    // |   |
    // 3---2
    EdgeListGraph g;
    g.V = 4;
    g.edges.push_back({0, 1, 1});
    g.edges.push_back({1, 2, 1});
    g.edges.push_back({2, 3, 1});
    g.edges.push_back({3, 0, 1});

    // 打印所有边
    cout << "Edge List:" << endl;
    for (const auto& e : g.edges) {
        cout << e.src << " -- " << e.dest 
             << " (weight: " << e.weight << ")" << endl;
    }

    return 0;
}

In [ ]:
main();

**输出：**
```
Edge List:
0 -- 1 (weight: 1)
1 -- 2 (weight: 1)
2 -- 3 (weight: 1)
3 -- 0 (weight: 1)
```

#### 2.3 复杂度分析

| 操作 | 时间复杂度 |
|------|-----------|
| 判断 (u,v) 是否有边 | O(E) — 需要遍历所有边 |
| 求某顶点的所有邻接点 | O(E) — 需要遍历所有边 |
| 添加一条边 | O(1) — 直接追加 |
| 空间复杂度 | O(E) |

> **适用场景**：Kruskal 算法（需要对边排序）。但对于一般用途，效率太低，很少单独使用。

---

### 3. 邻接矩阵法（6.2.1）

#### 3.1 核心思想

用一个 **V × V** 的二维数组 `A[i][j]` 来表示图：

- **无权图**：`A[i][j] = 1` 表示有边，`A[i][j] = 0` 表示无边。
- **带权图**：`A[i][j] = w` 表示边权为 w，`A[i][j] = ∞` 表示无边，`A[i][i] = 0`。

```
无向图：              邻接矩阵：
  0---1                 0  1  2  3
  |   |              0 [0  1  0  1]
  3---2              1 [1  0  1  0]
                     2 [0  1  0  1]
                     3 [1  0  1  0]

注意：无向图的邻接矩阵是 对称矩阵！
```

```
有向图：              邻接矩阵：
  0→1                   0  1  2
  ↑  ↓               0 [0  1  0]
  2←─┘               1 [0  0  1]
                     2 [1  0  0]

注意：有向图的邻接矩阵 不一定 对称！
```

#### 3.2 邻接矩阵的数学性质

这部分非常重要，考试常考：

> 以下计数与矩阵幂结论针对**固定顶点编号顺序的无权简单图的 0/1 邻接矩阵**。

**性质1**：无向图邻接矩阵第 i 行（或第 i 列）值为 1 的元素个数 = 顶点 i 的度。

**性质2**：有向图邻接矩阵第 i 行值为 1 的元素个数 = 顶点 i 的**出度**；第 i 列值为 1 的元素个数 = 顶点 i 的**入度**。

**性质3（矩阵幂的含义）**：设 A 为图 G 的邻接矩阵，则 **A^n[i][j]** 表示从顶点 i 到顶点 j 的**长度为 n 的走法数**。

> 🎯 **考试真题高频考点**：A² 中 `A²[i][j]` 的含义 = 从 i 经过恰好 2 条边到达 j 的走法数（允许重复顶点或边）。

#### 3.3 C++ 完整实现

In [ ]:
#include <iostream>
#include <vector>
#include <climits>
using namespace std;

In [ ]:
class AdjMatrixGraph {
private:
    int V;                          // 顶点数
    vector<vector<int>> matrix;     // 邻接矩阵
    bool isDirected;                // 是否有向图
    bool isWeighted;                // 是否带权图

public:
    // 构造函数
    AdjMatrixGraph(int vertices, bool directed = false, bool weighted = false) 
        : V(vertices), isDirected(directed), isWeighted(weighted) {
        // 初始化矩阵
        if (weighted) {
            // 带权图：初始化为 INT_MAX 表示无穷大，对角线为 0
            matrix.assign(V, vector<int>(V, INT_MAX));
            for (int i = 0; i < V; i++)
                matrix[i][i] = 0;
        } else {
            // 无权图：全部初始化为 0
            matrix.assign(V, vector<int>(V, 0));
        }
    }

    // 添加边
    void addEdge(int u, int v, int weight = 1) {
        if (u < 0 || u >= V || v < 0 || v >= V) {
            cout << "Error: vertex out of range!" << endl;
            return;
        }
        matrix[u][v] = weight;
        if (!isDirected) {
            matrix[v][u] = weight;  // 无向图：对称
        }
    }

    // 删除边
    void removeEdge(int u, int v) {
        int noEdge = isWeighted ? INT_MAX : 0;
        matrix[u][v] = noEdge;
        if (!isDirected) {
            matrix[v][u] = noEdge;
        }
    }

    // 判断边是否存在
    bool hasEdge(int u, int v) const {
        if (isWeighted)
            return u != v && matrix[u][v] != INT_MAX;
        return matrix[u][v] != 0;
    }

    // 获取边的权值
    int getWeight(int u, int v) const {
        return matrix[u][v];
    }

    // 求顶点的度（无向图）/ 出度（有向图）
    int outDegree(int v) const {
        int deg = 0;
        for (int j = 0; j < V; j++) {
            if (isWeighted) {
                if (j != v && matrix[v][j] != INT_MAX) deg++;
            } else {
                if (matrix[v][j] != 0) deg++;
            }
        }
        return deg;
    }

    // 求入度（有向图）
    int inDegree(int v) const {
        int deg = 0;
        for (int i = 0; i < V; i++) {
            if (isWeighted) {
                if (i != v && matrix[i][v] != INT_MAX) deg++;
            } else {
                if (matrix[i][v] != 0) deg++;
            }
        }
        return deg;
    }

    // 获取所有邻接点
    vector<int> getNeighbors(int v) const {
        vector<int> neighbors;
        for (int j = 0; j < V; j++) {
            if (isWeighted) {
                if (matrix[v][j] != INT_MAX && v != j) 
                    neighbors.push_back(j);
            } else {
                if (matrix[v][j] != 0) 
                    neighbors.push_back(j);
            }
        }
        return neighbors;
    }

    // 打印邻接矩阵
    void print() const {
        cout << "Adjacency Matrix (" << V << " vertices):" << endl;
        cout << "    ";
        for (int i = 0; i < V; i++) cout << i << "   ";
        cout << endl;

        for (int i = 0; i < V; i++) {
            cout << i << " [ ";
            for (int j = 0; j < V; j++) {
                if (isWeighted && matrix[i][j] == INT_MAX)
                    cout << "∞   ";
                else
                    cout << matrix[i][j] << "   ";
            }
            cout << "]" << endl;
        }
    }

    int getV() const { return V; }
};

In [ ]:
int main() {
    // ======== 示例1：无向无权图 ========
    cout << "===== 无向无权图 =====" << endl;
    AdjMatrixGraph g1(4, false, false);
    //  0 --- 1
    //  |     |
    //  3 --- 2
    g1.addEdge(0, 1);
    g1.addEdge(1, 2);
    g1.addEdge(2, 3);
    g1.addEdge(3, 0);
    g1.print();

    cout << "顶点0的度: " << g1.outDegree(0) << endl;
    cout << "边(0,1)存在? " << (g1.hasEdge(0, 1) ? "Yes" : "No") << endl;
    cout << "边(0,2)存在? " << (g1.hasEdge(0, 2) ? "Yes" : "No") << endl;

    cout << "顶点1的邻接点: ";
    for (int n : g1.getNeighbors(1)) cout << n << " ";
    cout << endl << endl;

    // ======== 示例2：有向带权图 ========
    cout << "===== 有向带权图 =====" << endl;
    AdjMatrixGraph g2(4, true, true);
    g2.addEdge(0, 1, 5);
    g2.addEdge(0, 3, 7);
    g2.addEdge(1, 2, 3);
    g2.addEdge(2, 0, 2);
    g2.addEdge(3, 2, 1);
    g2.print();

    cout << "顶点0: 出度=" << g2.outDegree(0) 
         << ", 入度=" << g2.inDegree(0) << endl;
    cout << "边<0,1>的权值: " << g2.getWeight(0, 1) << endl;

    return 0;
}

In [ ]:
main();

**输出：**
```
===== 无向无权图 =====
Adjacency Matrix (4 vertices):
    0   1   2   3   
0 [ 0   1   0   1   ]
1 [ 1   0   1   0   ]
2 [ 0   1   0   1   ]
3 [ 1   0   1   0   ]
顶点0的度: 2
边(0,1)存在? Yes
边(0,2)存在? No
顶点1的邻接点: 0 2 

===== 有向带权图 =====
Adjacency Matrix (4 vertices):
    0   1   2   3   
0 [ 0   5   ∞   7   ]
1 [ ∞   0   3   ∞   ]
2 [ 2   ∞   0   ∞   ]
3 [ ∞   ∞   1   0   ]
顶点0: 出度=2, 入度=1
边<0,1>的权值: 5
```

#### 3.4 邻接矩阵优缺点总结

| 方面 | 详情 |
|------|------|
| ✅ 判断边是否存在 | O(1) — 直接访问 `matrix[u][v]` |
| ✅ 求边的权值 | O(1) |
| ✅ 实现简单 | 二维数组，直观易懂 |
| ❌ 空间复杂度 | O(V²) — 与边数无关，稀疏图浪费严重 |
| ❌ 求所有邻接点 | O(V) — 必须扫描整行 |
| ❌ 添加/删除顶点 | 很麻烦，需要重建矩阵 |

> **一句话判断**：顶点少、边多（稠密图）→ 用邻接矩阵。顶点多、边少（稀疏图）→ 用邻接表。

---

### 4. 邻接表法（6.2.2）

#### 4.1 核心思想

邻接表的核心思想：**为每个顶点维护一个链表，链表中存放所有与该顶点邻接的顶点**。

```
图：                    邻接表：
  0---1                 0 → [1] → [3]
  |   |                 1 → [0] → [2]
  3---2                 2 → [1] → [3]
                        3 → [2] → [0]
```

它本质上是一个"**数组 + 链表**"的混合结构：
- **数组部分**：大小为 V 的数组，每个元素是一个链表的头指针。
- **链表部分**：每个链表节点存储一个邻接顶点的编号（和权值）。

```
有向图的邻接表：         
  0 → 1                0 → [1] → [3]     ← 出边表
  0 → 3                1 → [2]
  1 → 2                2 → [0]
  2 → 0                3 → [2]
  3 → 2
```

#### 4.2 C++ 完整实现

In [ ]:
#include <iostream>
#include <vector>
#include <list>
using namespace std;

// 邻接表节点

In [ ]:
struct AdjNode {
    int dest;       // 邻接顶点编号
    int weight;     // 权值
    AdjNode(int d, int w = 1) : dest(d), weight(w) {}
};

In [ ]:
class AdjListGraph {
private:
    int V;                              // 顶点数
    vector<list<AdjNode>> adjList;      // 邻接表
    bool isDirected;

public:
    AdjListGraph(int vertices, bool directed = false) 
        : V(vertices), isDirected(directed) {
        adjList.resize(V);
    }

    // 添加边
    void addEdge(int u, int v, int weight = 1) {
        adjList[u].push_back(AdjNode(v, weight));
        if (!isDirected) {
            adjList[v].push_back(AdjNode(u, weight));
        }
    }

    // 删除边
    void removeEdge(int u, int v) {
        adjList[u].remove_if([v](const AdjNode& node) {
            return node.dest == v;
        });
        if (!isDirected) {
            adjList[v].remove_if([u](const AdjNode& node) {
                return node.dest == u;
            });
        }
    }

    // 判断边是否存在 — O(degree(u))
    bool hasEdge(int u, int v) const {
        for (const auto& node : adjList[u]) {
            if (node.dest == v) return true;
        }
        return false;
    }

    // 获取所有邻接点 — O(degree(v))
    vector<int> getNeighbors(int v) const {
        vector<int> neighbors;
        for (const auto& node : adjList[v]) {
            neighbors.push_back(node.dest);
        }
        return neighbors;
    }

    // 求顶点的度/出度
    int outDegree(int v) const {
        return adjList[v].size();
    }

    // 求入度（有向图）— 需要遍历所有链表
    int inDegree(int v) const {
        int deg = 0;
        for (int i = 0; i < V; i++) {
            for (const auto& node : adjList[i]) {
                if (node.dest == v) deg++;
            }
        }
        return deg;
    }

    // 打印邻接表
    void print() const {
        cout << "Adjacency List (" << V << " vertices):" << endl;
        for (int i = 0; i < V; i++) {
            cout << i << " → ";
            bool first = true;
            for (const auto& node : adjList[i]) {
                if (!first) cout << " → ";
                cout << "[" << node.dest;
                if (node.weight != 1) cout << ",w=" << node.weight;
                cout << "]";
                first = false;
            }
            cout << endl;
        }
    }

    int getV() const { return V; }
    const list<AdjNode>& getAdj(int v) const { return adjList[v]; }
};

In [ ]:
int main() {
    // ======== 无向图 ========
    cout << "===== 无向图 =====" << endl;
    AdjListGraph g1(5, false);
    g1.addEdge(0, 1);
    g1.addEdge(0, 4);
    g1.addEdge(1, 2);
    g1.addEdge(1, 3);
    g1.addEdge(1, 4);
    g1.addEdge(2, 3);
    g1.addEdge(3, 4);
    g1.print();

    cout << "顶点1的度: " << g1.outDegree(1) << endl;
    cout << "顶点1的邻接点: ";
    for (int n : g1.getNeighbors(1)) cout << n << " ";
    cout << endl << endl;

    // ======== 有向带权图 ========
    cout << "===== 有向带权图 =====" << endl;
    AdjListGraph g2(4, true);
    g2.addEdge(0, 1, 5);
    g2.addEdge(0, 3, 7);
    g2.addEdge(1, 2, 3);
    g2.addEdge(2, 0, 2);
    g2.addEdge(3, 2, 1);
    g2.print();

    cout << "顶点0: 出度=" << g2.outDegree(0) 
         << ", 入度=" << g2.inDegree(0) << endl;

    return 0;
}

In [ ]:
main();

**输出：**
```
===== 无向图 =====
Adjacency List (5 vertices):
0 → [1] → [4]
1 → [0] → [2] → [3] → [4]
2 → [1] → [3]
3 → [1] → [2] → [4]
4 → [0] → [1] → [3]
顶点1的度: 4
顶点1的邻接点: 0 2 3 4 

===== 有向带权图 =====
Adjacency List (4 vertices):
0 → [1,w=5] → [3,w=7]
1 → [2,w=3]
2 → [0,w=2]
3 → [2,w=1]
顶点0: 出度=2, 入度=1
```

#### 4.3 邻接表优缺点总结

| 方面 | 详情 |
|------|------|
| ✅ 空间高效 | O(V + E) — 只存实际存在的边 |
| ✅ 遍历邻接点 | O(degree(v)) — 只访问实际邻接的顶点 |
| ✅ 添加边 | O(1) |
| ❌ 判断边是否存在 | O(degree(u)) — 需要遍历链表 |
| ❌ 求有向图的入度 | O(V + E) — 需要遍历所有链表 |

#### 4.4 邻接矩阵 vs 邻接表：对比总结

| 比较维度 | 邻接矩阵 | 邻接表 |
|----------|----------|--------|
| 空间 | O(V²) | O(V + E) |
| 判断边存在 | **O(1)** ✅ | O(degree) |
| 遍历所有邻接点 | O(V) | **O(degree)** ✅ |
| 适合 | 稠密图 | 稀疏图 |
| 表示唯一？ | 固定顶点编号顺序后**唯一** | **不唯一**（链表顺序可变） |

> **⚠️ 考试重点**：固定顶点编号顺序后，同一个图的邻接矩阵表示是**唯一**的；邻接表表示则是**不唯一**的（插入顺序不同，链表顺序不同）。

---

### 5. 十字链表（Orthogonal List）（6.2.3）

#### 5.1 为什么需要十字链表？

我们发现，用**邻接表**存储有向图时，有一个痛点：

- 求**出度**很方便：直接看顶点的邻接链表长度。
- 求**入度**很麻烦：需要遍历所有顶点的邻接链表。

为了解决这个问题，我们需要同时维护"出边"和"入边"两个链表。这就是**十字链表**的核心思想。

#### 5.2 核心结构

**弧节点（Arc Node）**：

```
+----------+----------+----------+----------+----------+
| tailvex  | headvex  |  hlink   |  tlink   |   info   |
+----------+----------+----------+----------+----------+
  弧尾(起点) 弧头(终点)  弧头相同    弧尾相同    权值等
                        的下一条弧  的下一条弧    附加信息
```

- `tailvex`：弧的起点（尾）
- `headvex`：弧的终点（头）
- `hlink`：指向弧头 (headvex) 相同的下一条弧 → 方便找某顶点的所有**入弧**
- `tlink`：指向弧尾 (tailvex) 相同的下一条弧 → 方便找某顶点的所有**出弧**

**顶点节点**：

```
+--------+------------+------------+
|  data  | firstin    | firstout   |
+--------+------------+------------+
  数据     第一条入弧    第一条出弧
```

#### 5.3 图示理解

```
有向图：0→1, 0→2, 1→2

顶点表:
  V0: firstin=NULL,  firstout → 弧(0,1)
  V1: firstin → 弧(0,1), firstout → 弧(1,2)
  V2: firstin → 弧(0,2), firstout = NULL

弧节点链接：
  弧(0,1): tailvex=0, headvex=1, tlink→弧(0,2), hlink=NULL
  弧(0,2): tailvex=0, headvex=2, tlink=NULL,     hlink→弧(1,2)
  弧(1,2): tailvex=1, headvex=2, tlink=NULL,     hlink=NULL

解读：
- V0 的出弧链 (tlink): 弧(0,1) → 弧(0,2) → NULL  → 出度=2
- V1 的入弧链 (hlink): 弧(0,1) → NULL             → 入度=1
- V2 的入弧链 (hlink): 弧(0,2) → 弧(1,2) → NULL   → 入度=2
```

#### 5.4 C++ 实现

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

// 弧节点

In [ ]:
struct ArcNode {
    int tailvex;        // 弧尾（起点）
    int headvex;        // 弧头（终点）
    ArcNode* hlink;     // 弧头相同的下一条弧（入弧链）
    ArcNode* tlink;     // 弧尾相同的下一条弧（出弧链）
    int weight;         // 权值

    ArcNode(int t, int h, int w = 1)
        : tailvex(t), headvex(h), hlink(nullptr), tlink(nullptr), weight(w) {}
};

In [ ]:
// 顶点节点
struct VexNode {
    int data;               // 顶点数据
    ArcNode* firstin;       // 第一条入弧
    ArcNode* firstout;      // 第一条出弧

    VexNode() : data(0), firstin(nullptr), firstout(nullptr) {}
};

In [ ]:
class OrthogonalListGraph {
private:
    int V;
    vector<VexNode> vexList;

public:
    OrthogonalListGraph(int vertices) : V(vertices) {
        vexList.resize(V);
        for (int i = 0; i < V; i++)
            vexList[i].data = i;
    }

    // 添加弧 <u, v>
    void addArc(int u, int v, int weight = 1) {
        ArcNode* arc = new ArcNode(u, v, weight);

        // 插入到 u 的出弧链（头插法）
        arc->tlink = vexList[u].firstout;
        vexList[u].firstout = arc;

        // 插入到 v 的入弧链（头插法）
        arc->hlink = vexList[v].firstin;
        vexList[v].firstin = arc;
    }

    // 求出度
    int outDegree(int v) const {
        int deg = 0;
        ArcNode* p = vexList[v].firstout;
        while (p) { deg++; p = p->tlink; }
        return deg;
    }

    // 求入度 — 同样 O(入度) 即可！
    int inDegree(int v) const {
        int deg = 0;
        ArcNode* p = vexList[v].firstin;
        while (p) { deg++; p = p->hlink; }
        return deg;
    }

    void print() const {
        cout << "Orthogonal List Graph:" << endl;
        for (int i = 0; i < V; i++) {
            // 出弧
            cout << "V" << i << " 出弧: ";
            ArcNode* p = vexList[i].firstout;
            while (p) {
                cout << "<" << p->tailvex << "," << p->headvex << "> ";
                p = p->tlink;
            }
            // 入弧
            cout << " | 入弧: ";
            p = vexList[i].firstin;
            while (p) {
                cout << "<" << p->tailvex << "," << p->headvex << "> ";
                p = p->hlink;
            }
            cout << endl;
        }
    }

    ~OrthogonalListGraph() {
        // 释放所有弧节点（通过出弧链遍历）
        for (int i = 0; i < V; i++) {
            ArcNode* p = vexList[i].firstout;
            while (p) {
                ArcNode* temp = p;
                p = p->tlink;
                delete temp;
            }
        }
    }
};

In [ ]:
int main() {
    OrthogonalListGraph g(3);
    // 0→1, 0→2, 1→2
    g.addArc(0, 1);
    g.addArc(0, 2);
    g.addArc(1, 2);
    g.print();

    for (int i = 0; i < 3; i++) {
        cout << "V" << i << ": 出度=" << g.outDegree(i)
             << ", 入度=" << g.inDegree(i) << endl;
    }

    return 0;
}

In [ ]:
main();

**输出：**
```
Orthogonal List Graph:
V0 出弧: <0,2> <0,1>  | 入弧: 
V1 出弧: <1,2>  | 入弧: <0,1> 
V2 出弧:  | 入弧: <1,2> <0,2> 
V0: 出度=2, 入度=0
V1: 出度=1, 入度=1
V2: 出度=0, 入度=2
```

> **十字链表要点**：
> - 专门用于**有向图**。
> - 求入度和出度都很方便，都是 O(度数)。
> - 空间复杂度 O(V + E)。

---

### 6. 邻接多重表（Adjacency Multilist）（6.2.4）

#### 6.1 为什么需要邻接多重表？

邻接表存储**无向图**时，每条边 `(u, v)` 会被存储两次（u 的链表里存一次，v 的链表里存一次）。这带来两个问题：

1. **空间浪费**：同一条边存了两份。
2. **操作不便**：删除一条边时，需要在两个链表中分别找到并删除，很麻烦。

邻接多重表的思想：**让每条边只用一个节点表示，但同时出现在两个顶点的链表中**。

#### 6.2 核心结构

**边节点**：

```
+--------+--------+--------+--------+--------+
| mark   | ivex   | ilink  | jvex   | jlink  |
+--------+--------+--------+--------+--------+
  标记     顶点i    i的下     顶点j    j的下
 (已访问)          一条边              一条边
```

- `ivex, jvex`：这条边连接的两个顶点。
- `ilink`：指向下一条依附于顶点 ivex 的边。
- `jlink`：指向下一条依附于顶点 jvex 的边。
- `mark`：标记该边是否被访问过（遍历时有用）。

**顶点节点**：

```
+--------+-----------+
|  data  | firstedge |
+--------+-----------+
  数据     第一条边
```

#### 6.3 图示理解

```
无向图：0-1, 0-2, 1-2

边节点：
  E0: ivex=0, jvex=1   （边 0-1）
  E1: ivex=0, jvex=2   （边 0-2）
  E2: ivex=1, jvex=2   （边 1-2）

顶点链接：
  V0.firstedge → E0
    E0.ilink(依附于0的下一条边) → E1
    E1.ilink(依附于0的下一条边) → NULL

  V1.firstedge → E0
    E0.jlink(依附于1的下一条边) → E2
    E2.ilink(依附于1的下一条边) → NULL

  V2.firstedge → E1
    E1.jlink(依附于2的下一条边) → E2
    E2.jlink(依附于2的下一条边) → NULL
```

#### 6.4 C++ 简要实现

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

// 边节点

In [ ]:
struct EdgeNode {
    bool mark;          // 访问标记
    int ivex, jvex;     // 两个顶点
    EdgeNode* ilink;    // 依附于 ivex 的下一条边
    EdgeNode* jlink;    // 依附于 jvex 的下一条边
    int weight;

    EdgeNode(int i, int j, int w = 1)
        : mark(false), ivex(i), jvex(j), 
          ilink(nullptr), jlink(nullptr), weight(w) {}
};

In [ ]:
struct VNode {
    int data;
    EdgeNode* firstedge;
    VNode() : data(0), firstedge(nullptr) {}
};

In [ ]:
class AdjMultilistGraph {
private:
    int V;
    vector<VNode> vexList;

public:
    AdjMultilistGraph(int vertices) : V(vertices) {
        vexList.resize(V);
        for (int i = 0; i < V; i++)
            vexList[i].data = i;
    }

    void addEdge(int i, int j, int w = 1) {
        EdgeNode* e = new EdgeNode(i, j, w);

        // 头插法插入到 i 的边链
        e->ilink = vexList[i].firstedge;
        vexList[i].firstedge = e;

        // 头插法插入到 j 的边链
        e->jlink = vexList[j].firstedge;
        vexList[j].firstedge = e;
    }

    // 遍历某顶点的所有邻接边
    void printEdgesOf(int v) const {
        cout << "V" << v << " 的边: ";
        EdgeNode* p = vexList[v].firstedge;
        while (p) {
            cout << "(" << p->ivex << "," << p->jvex << ") ";
            // 关键：判断当前顶点是 ivex 还是 jvex，决定走哪个链
            if (p->ivex == v)
                p = p->ilink;
            else
                p = p->jlink;
        }
        cout << endl;
    }

    void print() const {
        cout << "Adjacency Multilist Graph:" << endl;
        for (int i = 0; i < V; i++) {
            printEdgesOf(i);
        }
    }
};

In [ ]:
int main() {
    AdjMultilistGraph g(3);
    g.addEdge(0, 1);
    g.addEdge(0, 2);
    g.addEdge(1, 2);
    g.print();
    return 0;
}

In [ ]:
main();

**输出：**
```
Adjacency Multilist Graph:
V0 的边: (0,2) (0,1) 
V1 的边: (1,2) (0,1) 
V2 的边: (1,2) (0,2) 
```

#### 6.5 四种存储方式总览

| 存储方式 | 适用 | 空间 | 特点 |
|----------|------|------|------|
| **邻接矩阵** | 通用 | O(V²) | 判边 O(1)，稠密图首选 |
| **邻接表** | 通用 | O(V+E) | 稀疏图首选，最常用 |
| **十字链表** | 有向图 | O(V+E) | 入度出度都方便 |
| **邻接多重表** | 无向图 | O(V+E) | 每条边只存一份，删边方便 |

> **考试建议**：邻接矩阵和邻接表必须**完全掌握并手写代码**；十字链表和邻接多重表**理解结构和用途**即可，能画图说明。

---

### 7. 图的基本操作（6.2.5）

无论使用哪种存储方式，图都需要支持以下基本操作：

| 操作 | 说明 | 邻接矩阵 | 邻接表 |
|------|------|----------|--------|
| `Adjacent(G, x, y)` | 判断 (x,y) 是否有边 | O(1) | O(degree) |
| `Neighbors(G, x)` | 列出 x 的所有邻接点 | O(V) | O(degree) |
| `InsertVertex(G, x)` | 插入顶点 | O(V) 需扩展 | O(1) |
| `DeleteVertex(G, x)` | 删除顶点 | O(V²) | O(V+E) |
| `AddEdge(G, x, y)` | 添加边 | O(1) | O(1) |
| `RemoveEdge(G, x, y)` | 删除边 | O(1) | O(degree) |
| `FirstNeighbor(G, x)` | x 的第一个邻接点 | O(V) 扫描行 | O(1) 链表头 |
| `NextNeighbor(G, x, y)` | x 的邻接点中，y 之后的下一个 | O(V) | O(degree) |
| `Get_edge_value(G, x, y)` | 获取边权 | O(1) | O(degree) |
| `Set_edge_value(G, x, y, v)` | 设置边权 | O(1) | O(degree) |

> **这张表格是选择存储方式的核心依据**。根据你的算法需要哪些操作频繁，选择对应的存储结构。

---

## 第三部分：图的遍历

图的遍历是理解图结构和许多图算法的重要基础。最短路径、拓扑排序、连通分量等都是后续主题，但并非全部直接由 BFS/DFS 推导而来。

与树不同，图可能有**环**或多条到达同一顶点的路径，所以遍历必须维护已访问状态（数组、集合或颜色标记均可），避免重复访问；从单一起点只能覆盖其可达分量，非连通图还需要外层循环。

---

### 8. 广度优先搜索 BFS（6.3.1）

#### 8.1 核心思想

BFS 的思想就像**"水波扩散"**或**"层层推进"**：

1. 从起始顶点出发，先访问所有距离为 1 的顶点；
2. 再访问所有距离为 2 的顶点；
3. 以此类推……

> 如果你学过树的"层序遍历"，BFS 就是层序遍历在图上的推广。

**核心工具**：**队列（Queue）**

#### 8.2 BFS 算法步骤（伪代码思路）

```
BFS(图G, 起始顶点s):
    创建 visited[] 数组，全部初始化为 false
    创建一个空队列 Q
    
    visited[s] = true     // 标记起始顶点
    Q.push(s)             // 起始顶点入队
    
    while Q 不为空:
        u = Q.front()     // 取队首
        Q.pop()           // 出队
        访问/处理 u       // 比如打印
        
        for 每个 u 的邻接点 v:
            if visited[v] == false:
                visited[v] = true   // 先标记再入队！
                Q.push(v)
```

#### 8.3 手动模拟 BFS

```
图：
    0 --- 1 --- 2
    |           |
    3 --- 4 --- 5
          |
          6

邻接表（假设邻接点按编号升序排列）：
    0 → [1, 3]
    1 → [0, 2]
    2 → [1, 5]
    3 → [0, 4]
    4 → [3, 5, 6]
    5 → [2, 4]
    6 → [4]

从顶点 0 开始 BFS：

Step | Queue (front→back) | 出队 | 访问 | 新入队
-----|---------------------|------|------|-------
  0  | [0]                 |  -   |  0   |  -
  1  | [1, 3]              |  0   |  0   | 1, 3
  2  | [3, 2]              |  1   |  1   | 2
  3  | [2, 4]              |  3   |  3   | 4
  4  | [4, 5]              |  2   |  2   | 5
  5  | [5, 6]              |  4   |  4   | 6
  6  | [6]                 |  5   |  5   | -
  7  | []                  |  6   |  6   | -

BFS 遍历序列：0, 1, 3, 2, 4, 5, 6

层次划分：
  第0层(距离0): {0}
  第1层(距离1): {1, 3}
  第2层(距离2): {2, 4}
  第3层(距离3): {5, 6}
```

#### 8.4 C++ 完整实现

In [ ]:
#include <iostream>
#include <vector>
#include <list>
#include <queue>
using namespace std;

In [ ]:
class Graph {
private:
    int V;
    vector<list<int>> adj;
    bool isDirected;

public:
    Graph(int v, bool directed = false) : V(v), isDirected(directed) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
        if (!isDirected) adj[v].push_back(u);
    }

    // ==================== BFS ====================
    // 从单个顶点出发的 BFS
    void BFS(int start) const {
        vector<bool> visited(V, false);
        queue<int> q;

        visited[start] = true;
        q.push(start);

        cout << "BFS from vertex " << start << ": ";
        while (!q.empty()) {
            int u = q.front();
            q.pop();
            cout << u << " ";

            // 遍历 u 的所有邻接点
            for (int v : adj[u]) {
                if (!visited[v]) {
                    visited[v] = true;  // 入队前标记！
                    q.push(v);
                }
            }
        }
        cout << endl;
    }

    // 完整的 BFS（处理非连通图）
    void BFS_Complete() const {
        vector<bool> visited(V, false);
        cout << "Complete BFS: ";

        for (int i = 0; i < V; i++) {
            if (!visited[i]) {
                // 对每个未访问的连通分量执行 BFS
                queue<int> q;
                visited[i] = true;
                q.push(i);

                while (!q.empty()) {
                    int u = q.front();
                    q.pop();
                    cout << u << " ";

                    for (int v : adj[u]) {
                        if (!visited[v]) {
                            visited[v] = true;
                            q.push(v);
                        }
                    }
                }
                cout << "| ";  // 分隔不同连通分量
            }
        }
        cout << endl;
    }

    // BFS 求最短路径（无权图）
    vector<int> BFS_ShortestPath(int start) const {
        vector<int> dist(V, -1);    // -1 表示不可达
        vector<int> parent(V, -1);  // 记录路径
        queue<int> q;

        dist[start] = 0;
        q.push(start);

        while (!q.empty()) {
            int u = q.front();
            q.pop();

            for (int v : adj[u]) {
                if (dist[v] == -1) {  // 未访问
                    dist[v] = dist[u] + 1;
                    parent[v] = u;
                    q.push(v);
                }
            }
        }
        return dist;
    }

    // 打印从 start 到 end 的最短路径
    void printPath(int start, int end) const {
        vector<int> dist(V, -1);
        vector<int> parent(V, -1);
        queue<int> q;

        dist[start] = 0;
        q.push(start);

        while (!q.empty()) {
            int u = q.front();
            q.pop();
            for (int v : adj[u]) {
                if (dist[v] == -1) {
                    dist[v] = dist[u] + 1;
                    parent[v] = u;
                    q.push(v);
                }
            }
        }

        if (dist[end] == -1) {
            cout << "No path from " << start << " to " << end << endl;
            return;
        }

        // 回溯路径
        vector<int> path;
        for (int v = end; v != -1; v = parent[v])
            path.push_back(v);

        cout << "Shortest path (" << start << " → " << end 
             << "), length=" << dist[end] << ": ";
        for (int i = path.size() - 1; i >= 0; i--) {
            cout << path[i];
            if (i > 0) cout << " → ";
        }
        cout << endl;
    }

    // BFS 生成树
    void BFS_Tree(int start) const {
        vector<bool> visited(V, false);
        queue<int> q;

        visited[start] = true;
        q.push(start);

        cout << "BFS Tree edges from " << start << ": ";
        while (!q.empty()) {
            int u = q.front();
            q.pop();

            for (int v : adj[u]) {
                if (!visited[v]) {
                    visited[v] = true;
                    q.push(v);
                    cout << "(" << u << "," << v << ") ";  // 树边
                }
            }
        }
        cout << endl;
    }

    int getV() const { return V; }
};

In [ ]:
int main() {
    Graph g(7, false);
    //     0 --- 1 --- 2
    //     |           |
    //     3 --- 4 --- 5
    //           |
    //           6
    g.addEdge(0, 1);
    g.addEdge(1, 2);
    g.addEdge(0, 3);
    g.addEdge(3, 4);
    g.addEdge(4, 5);
    g.addEdge(2, 5);
    g.addEdge(4, 6);

    // 1. 基本 BFS
    g.BFS(0);

    // 2. 最短路径
    vector<int> dist = g.BFS_ShortestPath(0);
    cout << "Distances from 0: ";
    for (int i = 0; i < g.getV(); i++)
        cout << i << "=" << dist[i] << " ";
    cout << endl;

    // 3. 打印具体路径
    g.printPath(0, 6);
    g.printPath(0, 5);

    // 4. BFS 生成树
    g.BFS_Tree(0);

    cout << endl;

    // 5. 非连通图的完整 BFS
    Graph g2(8, false);
    g2.addEdge(0, 1);
    g2.addEdge(0, 2);
    g2.addEdge(1, 2);
    // 3-4-5 是另一个连通分量
    g2.addEdge(3, 4);
    g2.addEdge(4, 5);
    // 6-7 是第三个连通分量
    g2.addEdge(6, 7);
    g2.BFS_Complete();

    return 0;
}

In [ ]:
main();

**输出：**
```
BFS from vertex 0: 0 1 3 2 4 5 6 
Distances from 0: 0=0 1=1 2=2 3=1 4=2 5=3 6=3 
Shortest path (0 → 6), length=3: 0 → 3 → 4 → 6
Shortest path (0 → 5), length=3: 0 → 1 → 2 → 5
BFS Tree edges from 0: (0,1) (0,3) (1,2) (3,4) (4,5) (4,6) 

Complete BFS: 0 1 2 | 3 4 5 | 6 7 | 
```

#### 8.5 BFS 的关键性质

| 性质 | 说明 |
|------|------|
| **时间复杂度** | 邻接表：**O(V + E)**；邻接矩阵：**O(V²)** |
| **空间复杂度** | O(V)（visited 数组 + 队列） |
| **最短路径** | BFS 可以求**无权图**的最短路径（距离 = 层数） |
| **BFS 生成树** | BFS 首次发现顶点时选出的树边构成一棵生成树（连通图）或生成森林（非连通图） |
| **入队时标记** | visited 在**入队时**标记为 true，不是出队时！否则同一顶点可能被重复入队 |

> **⚠️ 极易错点**：`visited[v] = true` 必须在 `q.push(v)` **之前或同时**执行，绝不能等到出队时才标记。如果在出队时才标记，一个顶点可能被多次入队，导致重复访问和性能退化。

---

### 9. 深度优先搜索 DFS（6.3.2）

#### 9.1 核心思想

DFS 的思想就像**"走迷宫"**：

1. 从起点出发，沿着一条路一直走到底；
2. 走不动了（没有未访问的邻接点），就**回退（回溯）**到上一个顶点；
3. 从那个顶点再尝试另一条路；
4. 直到所有可达顶点都被访问。

> 如果你学过树的"先序遍历"，DFS 就是先序遍历在图上的推广。

**核心工具**：**栈（Stack）**——递归本身就用了系统调用栈。

#### 9.2 DFS 算法步骤

**递归版（最常用）：**

```
DFS(图G, 顶点u, visited[]):
    visited[u] = true
    访问/处理 u
    
    for 每个 u 的邻接点 v:
        if visited[v] == false:
            DFS(G, v, visited)   // 递归深入
```

**非递归版（用显式栈）：**

```
DFS_Iterative(图G, 起始顶点s):
    创建 visited[] 数组，全部初始化为 false
    创建一个空栈 S
    
    S.push(s)
    
    while S 不为空:
        u = S.top()
        S.pop()
        
        if visited[u] == false:
            visited[u] = true
            访问/处理 u
            
            for 每个 u 的邻接点 v (逆序压栈以保证顺序):
                if visited[v] == false:
                    S.push(v)
```

#### 9.3 手动模拟 DFS

```
图（同 BFS 的例子）：
    0 --- 1 --- 2
    |           |
    3 --- 4 --- 5
          |
          6

邻接表（邻接点按编号升序）：
    0 → [1, 3]
    1 → [0, 2]
    2 → [1, 5]
    3 → [0, 4]
    4 → [3, 5, 6]
    5 → [2, 4]
    6 → [4]

从顶点 0 开始 DFS（递归）：

DFS(0):  访问 0, 看邻接点 {1, 3}
  └→ DFS(1):  访问 1, 看邻接点 {0, 2}. 0 已访问
     └→ DFS(2):  访问 2, 看邻接点 {1, 5}. 1 已访问
        └→ DFS(5):  访问 5, 看邻接点 {2, 4}. 2 已访问
           └→ DFS(4):  访问 4, 看邻接点 {3, 5, 6}. 5 已访问
              └→ DFS(3):  访问 3, 看邻接点 {0, 4}. 都已访问
                 回溯到 4
              └→ DFS(6):  访问 6, 看邻接点 {4}. 4 已访问
                 回溯到 4
              回溯到 5
           回溯到 2
        回溯到 1
     回溯到 0

DFS 遍历序列：0 → 1 → 2 → 5 → 4 → 3 → 6

对比 BFS 序列：0 → 1 → 3 → 2 → 4 → 5 → 6
```

**直觉对比**：
- **BFS**：像水面涟漪，一层一层扩展。
- **DFS**：像一根绳子，先一路深入到底，再回来探索分支。

#### 9.4 C++ 完整实现

In [ ]:
#include <iostream>
#include <vector>
#include <list>
#include <stack>
using namespace std;

In [ ]:
class Graph {
private:
    int V;
    vector<list<int>> adj;
    bool isDirected;

    // ===== DFS 递归辅助函数 =====
    void DFS_Recursive_Helper(int u, vector<bool>& visited) const {
        visited[u] = true;
        cout << u << " ";

        for (int v : adj[u]) {
            if (!visited[v]) {
                DFS_Recursive_Helper(v, visited);
            }
        }
    }

    // 带时间戳的 DFS（理解 DFS 树的关键）
    void DFS_Timestamp_Helper(int u, vector<bool>& visited,
                              vector<int>& disc, vector<int>& finish,
                              int& time) const {
        visited[u] = true;
        disc[u] = ++time;    // 发现时间
        cout << u << "(d=" << disc[u] << ") ";

        for (int v : adj[u]) {
            if (!visited[v]) {
                DFS_Timestamp_Helper(v, visited, disc, finish, time);
            }
        }
        finish[u] = ++time;  // 完成时间
    }

public:
    Graph(int v, bool directed = false) : V(v), isDirected(directed) {
        adj.resize(V);
    }

    void addEdge(int u, int v) {
        adj[u].push_back(v);
        if (!isDirected) adj[v].push_back(u);
    }

    // ==================== DFS 递归版 ====================
    void DFS_Recursive(int start) const {
        vector<bool> visited(V, false);
        cout << "DFS (Recursive) from " << start << ": ";
        DFS_Recursive_Helper(start, visited);
        cout << endl;
    }

    // ==================== DFS 非递归版 ====================
    void DFS_Iterative(int start) const {
        vector<bool> visited(V, false);
        stack<int> s;
        s.push(start);

        cout << "DFS (Iterative) from " << start << ": ";
        while (!s.empty()) {
            int u = s.top();
            s.pop();

            if (visited[u]) continue;  // 跳过已访问
            visited[u] = true;
            cout << u << " ";

            // 逆序压栈，保证先访问编号小的
            // 将邻接点收集后逆序压入
            vector<int> neighbors(adj[u].begin(), adj[u].end());
            for (int i = static_cast<int>(neighbors.size()) - 1; i >= 0; i--) {
                if (!visited[neighbors[i]]) {
                    s.push(neighbors[i]);
                }
            }
        }
        cout << endl;
    }

    // ==================== 完整 DFS（处理非连通图） ====================
    void DFS_Complete() const {
        vector<bool> visited(V, false);
        cout << "Complete DFS: ";
        for (int i = 0; i < V; i++) {
            if (!visited[i]) {
                DFS_Recursive_Helper(i, visited);
                cout << "| ";  // 分隔不同连通分量
            }
        }
        cout << endl;
    }

    // ==================== 带时间戳的 DFS ====================
    void DFS_Timestamp(int start) const {
        vector<bool> visited(V, false);
        vector<int> disc(V, 0), finish(V, 0);
        int time = 0;

        cout << "DFS with timestamps from " << start << ":" << endl;
        DFS_Timestamp_Helper(start, visited, disc, finish, time);
        cout << endl;

        cout << "Vertex | Discover | Finish" << endl;
        cout << "-------|----------|-------" << endl;
        for (int i = 0; i < V; i++) {
            if (disc[i] > 0)
                cout << "   " << i << "   |    " << disc[i] 
                     << "     |   " << finish[i] << endl;
        }
    }

    // ==================== DFS 应用：判断是否有环 ====================
    bool hasCycle_DFS_Helper(int u, vector<int>& color, 
                             vector<int>& parent) const {
        // color: 0=白(未访问), 1=灰(正在访问), 2=黑(已完成)
        color[u] = 1;  // 标记为灰色

        for (int v : adj[u]) {
            if (color[v] == 1) {
                // 遇到灰色顶点 = 回边 = 有环
                if (isDirected || v != parent[u]) {
                    return true;
                }
            }
            if (color[v] == 0) {
                parent[v] = u;
                if (hasCycle_DFS_Helper(v, color, parent))
                    return true;
            }
        }

        color[u] = 2;  // 标记为黑色
        return false;
    }

    bool hasCycle() const {
        vector<int> color(V, 0);
        vector<int> parent(V, -1);

        for (int i = 0; i < V; i++) {
            if (color[i] == 0) {
                if (hasCycle_DFS_Helper(i, color, parent))
                    return true;
            }
        }
        return false;
    }

    // ==================== DFS 应用：求连通分量 ====================
    int countConnectedComponents() const {
        vector<bool> visited(V, false);
        int count = 0;
        for (int i = 0; i < V; i++) {
            if (!visited[i]) {
                DFS_Recursive_Helper(i, visited);
                count++;
            }
        }
        return count;
    }

    int getV() const { return V; }
};

In [ ]:
int main() {
    Graph g(7, false);
    //     0 --- 1 --- 2
    //     |           |
    //     3 --- 4 --- 5
    //           |
    //           6
    g.addEdge(0, 1);
    g.addEdge(1, 2);
    g.addEdge(0, 3);
    g.addEdge(3, 4);
    g.addEdge(4, 5);
    g.addEdge(2, 5);
    g.addEdge(4, 6);

    // 1. DFS 递归
    g.DFS_Recursive(0);

    // 2. DFS 非递归
    g.DFS_Iterative(0);

    // 3. 带时间戳的 DFS
    g.DFS_Timestamp(0);

    // 4. 判断是否有环
    cout << endl << "Has cycle? " << (g.hasCycle() ? "Yes" : "No") << endl;

    // 5. 非连通图
    cout << endl << "===== 非连通图 =====" << endl;
    Graph g2(8, false);
    g2.addEdge(0, 1);
    g2.addEdge(0, 2);
    g2.addEdge(3, 4);
    g2.addEdge(4, 5);
    g2.addEdge(6, 7);
    g2.DFS_Complete();
    // （打印会输出到 cout 内部，额外打印计数）

    cout << endl << "===== 有向图判环 =====" << endl;
    Graph g3(4, true);
    g3.addEdge(0, 1);
    g3.addEdge(1, 2);
    g3.addEdge(2, 3);
    cout << "无环有向图 - Has cycle? " << (g3.hasCycle() ? "Yes" : "No") << endl;

    Graph g4(4, true);
    g4.addEdge(0, 1);
    g4.addEdge(1, 2);
    g4.addEdge(2, 0);  // 形成环
    cout << "有环有向图 - Has cycle? " << (g4.hasCycle() ? "Yes" : "No") << endl;

    return 0;
}

In [ ]:
main();

**输出：**
```
DFS (Recursive) from 0: 0 1 2 5 4 3 6 
DFS (Iterative) from 0: 0 1 2 5 4 3 6 
DFS with timestamps from 0:
0(d=1) 1(d=2) 2(d=3) 5(d=4) 4(d=5) 3(d=6) 6(d=8) 
Vertex | Discover | Finish
-------|----------|-------
   0   |    1     |   14
   1   |    2     |   13
   2   |    3     |   12
   3   |    6     |   7
   4   |    5     |   11
   5   |    4     |   10
   6   |    8     |   9

Has cycle? Yes

===== 非连通图 =====
Complete DFS: 0 1 2 | 3 4 5 | 6 7 | 

===== 有向图判环 =====
无环有向图 - Has cycle? No
有环有向图 - Has cycle? Yes
```

#### 9.5 DFS 时间戳的深层含义

DFS 的"发现时间 (d)"和"完成时间 (f)"构成的区间有着重要的嵌套关系——**括号定理**：

```
对任意两个顶点 u 和 v：
- 若 u 是 v 的祖先：[d[u], f[u]] 完全包含 [d[v], f[v]]
- 若 u 和 v 无祖先关系：两个区间完全不相交

上例中：
  顶点0: [1, 14]  完全包含所有其他顶点 → 0 是所有顶点的祖先
  顶点1: [2, 13]  包含 顶点2 [3,12] → 1 是 2 的祖先
  顶点3: [6, 7]   不包含任何其他 → 3 是叶子
```

#### 9.6 DFS 边的分类（有向图）

在 DFS 过程中，图中的边可以被分为四类：

| 边类型 | 判断条件 | 含义 |
|--------|----------|------|
| **树边 (Tree Edge)** | v 是白色（未发现） | DFS 树上的边 |
| **回边 (Back Edge)** | v 是灰色（正在处理） | 指向祖先 → **存在环** |
| **前向边 (Forward Edge)** | v 是黑色且 d[u] < d[v] | 指向后代（非树边） |
| **交叉边 (Cross Edge)** | v 是黑色且 d[u] > d[v] | 指向其他分支的顶点 |

> **核心结论**：**有向图有环 ⟺ DFS 发现回边**。这是拓扑排序可行性的判断依据。

---

### 10. BFS vs DFS 总结对比

| 维度 | BFS | DFS |
|------|-----|-----|
| **数据结构** | 队列（Queue） | 栈（Stack）/ 递归 |
| **空间复杂度** | O(V)（队列最大宽度） | O(V)（递归深度/栈深度） |
| **时间复杂度** | 邻接表 O(V+E)，矩阵 O(V²) | 邻接表 O(V+E)，矩阵 O(V²) |
| **遍历特点** | 层层推进 | 一路到底再回溯 |
| **最短路径** | ✅ 可求无权图最短路径 | 标准 DFS 遍历不保证无权图最短路径；穷举路径可以求解，但不再是 O(V+E) 的 DFS 遍历算法 |
| **拓扑排序** | ✅ Kahn 算法 | ✅ 逆后序 |
| **判环** | ✅ 有向图（Kahn） | ✅ 回边检测 |
| **连通分量** | ✅ | ✅ |
| **生成树** | BFS 生成树（宽度优先） | DFS 生成树（深度优先） |
| **实际偏好** | 求最短路径时首选 | 判环、拓扑排序、回溯问题首选 |

---

### 11. 综合练习题

#### 练习1：手动画图

给定如下有向图，请分别画出其**邻接矩阵**和**邻接表**。

```
顶点：{A, B, C, D, E}
边：A→B, A→D, B→C, C→E, D→C, E→D
```

<details>
<summary>点击查看答案</summary>

**邻接矩阵（A=0, B=1, C=2, D=3, E=4）：**
```
    A  B  C  D  E
A [ 0  1  0  1  0 ]
B [ 0  0  1  0  0 ]
C [ 0  0  0  0  1 ]
D [ 0  0  1  0  0 ]
E [ 0  0  0  1  0 ]
```

**邻接表：**
```
A → [B] → [D]
B → [C]
C → [E]
D → [C]
E → [D]
```
</details>

#### 练习2：手动模拟 BFS 和 DFS

对练习1的图，分别从顶点 A 开始执行 BFS 和 DFS，写出遍历序列。

<details>
<summary>点击查看答案</summary>

**BFS（从 A 出发）：**
```
队列变化：
[A] → 出A，入B,D → [B,D]
[B,D] → 出B，入C → [D,C]
[D,C] → 出D，C已在队列中 → [C]
[C] → 出C，入E → [E]
[E] → 出E，D已访问 → []

BFS 序列：A, B, D, C, E
```

**DFS（从 A 出发，递归）：**
```
DFS(A): 访问A → 邻接点 B, D
  DFS(B): 访问B → 邻接点 C
    DFS(C): 访问C → 邻接点 E
      DFS(E): 访问E → 邻接点 D
        DFS(D): 访问D → 邻接点 C(已访问)
        回溯

DFS 序列：A, B, C, E, D
```
</details>

#### 练习3：算法分析题

问：对于一个有 V 个顶点、E 条边的**无向连通图**：
1. BFS 生成树有多少条边？
2. 非树边有多少条？
3. 如果该图的 BFS 生成树和 DFS 生成树完全相同，说明什么？

<details>
<summary>点击查看答案</summary>

1. 生成树边数 = **V - 1**（任何树都是 V 个顶点 V-1 条边）
2. 非树边 = E - (V-1) = **E - V + 1**
3. 如果 BFS 树 = DFS 树，说明图本身就是一棵树（无环、连通），即 E = V - 1。因为无向图 DFS 的非树边只会连向祖先；若存在这样的额外边，就会让 BFS 得到更短层次，二者不可能完全相同。
</details>

---

### 12. 本章小结

```
图的知识体系导图：

                         图 (Graph)
                            │
            ┌───────────────┼───────────────┐
            │               │               │
         基本概念         存储结构          遍历算法
            │               │               │
    ┌───────┼──────┐    ┌───┼───┐       ┌───┴───┐
    │       │      │    │   │   │       │       │
  有向图  无向图  术语  邻接  邻接  十字   BFS     DFS
                   │   矩阵  表   链表/   │       │
              度/路径/      多重表  │       │
              连通性              │       │
                            无权图    判环/
                            最短路   拓扑排序/
                                   连通分量
```

**核心记忆点**：

1. **存储选择**：稠密→矩阵，稀疏→邻接表。
2. **BFS 用队列**，逐层扩展，可求无权图最短路。
3. **DFS 用栈/递归**，一路到底再回溯，可判环。
4. **两者时间复杂度相同**：邻接表 O(V+E)，邻接矩阵 O(V²)。
5. **已访问状态**（数组、集合或颜色标记）是图遍历区别于树遍历的关键之一——避免重复访问。

---

> 📚 **下一阶段预告**：掌握了图的存储和遍历后，你将进入图论的高级应用——**最小生成树（Prim/Kruskal）、最短路径（Dijkstra/Floyd）、拓扑排序、关键路径**等经典算法。它们都依赖图的表示与基本遍历思想，但不都是 BFS/DFS 的直接延伸。